# 第一个网络爬虫程序

需求：抓取华中杯官网 http://hzbmmc.com/index.html 大赛简介内容
* 分析过程
* 分析大赛简介内容在网页源代码中是否出现

In [ ]:
import requests as req

## 定义抓取网页地址

In [ ]:
url = 'http://hzbmmc.com/index.html'

## 发送请求

In [ ]:
response = req.get(url)

## 查看响应属性

In [ ]:
#查看响应编码
response.encoding

In [ ]:
#查看网页字符串源代码
response.text

In [ ]:
#查看网页字节流编码
response.content

In [ ]:
#查看状态响应码 只有状态码是200时才是正常访问
response.status_code

## 修改网页编码

In [ ]:
#将网页编码方式修改为UTF-8
response.encoding='UTF-8'

In [ ]:
#再次查看网页源码，发现中文正常显示了
response.text

## 保存网页源码到本地

In [ ]:
with open('hzb.text','w',encoding='utf-8')as f:
    f.write(response.text)

## 保存图片到本地

In [ ]:
img_url='http://hzbmmc.com/images/v2_qmyf0e.png'

In [ ]:
img_response=req.get(img_url)

In [ ]:
with open('hzb.png','wb') as f:
    f.write(img_response.content)

In [ ]:
#保存海报图片到本地
poster_url='http://hzbmmc.com:9000/hzb/temp/%E9%A6%96%E5%9B%BE_1644910802548.jpg'

In [ ]:
poster_response=req.get(poster_url)

In [ ]:
with open('hzb s poster.jpg','wb') as f:
    f.write(poster_response.content)

## 数据解析

### 使用正则表达式解析

In [ ]:
import re

In [ ]:
pattern=' <meta name="description" content="(.*?)">'

In [ ]:
re.findall(pattern,response.text)

In [ ]:
#匹配图片地址
pattern1= '<img src="(.*?)" class="logo-img" />'

In [ ]:
re.findall(pattern1,response.text)

In [ ]:
#在路径前面加上域名拼接成完整路径
'hzbmmc.com'+re.findall(pattern1,response.text)[0][1:]

### 使用Xpath语法进行解析

#### 将HTML字符串转化为XML格式数据

In [ ]:
from lxml import etree 

In [ ]:
html = etree.HTML(response.text)

In [ ]:
#取出大赛简介内容
html.xpath('//meta[@name="description"]/@content')

In [ ]:
#取出图片地址
html.xpath('//img[@class="logo-img"]/@src')

# 抓取豆瓣短评
- https://movie.douban.com/subject/26654184/comments?status=P

## 定义url地址

In [ ]:
url="https://movie.douban.com/subject/26654184/comments?status=P"

## 发送请求

In [ ]:
res=req.get(url)

In [ ]:
res.status_code

这里状态码出现了异常，说明使用requests发送的请求被识别成了爬虫


## 为get请求添加请求头

In [ ]:
headers={
    'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:106.0) Gecko/20100101 Firefox/106.0'
}

In [ ]:
res=req.get(url,headers=headers)

In [ ]:
res.status_code

In [ ]:
res.text

## 将短评解析出来

In [ ]:
import re

In [ ]:
pattern='<span class="short">(.*?)</span>'

In [ ]:
#flags：一般这个字段的值是不需要修改的，当我们匹配的字符串中包含换行符号时才需要对其内容进行修改为re.S
re.findall(pattern,res.text,flags=re.S)

In [ ]:
from lxml import etree

In [ ]:
html=etree.HTML(res.text)

In [ ]:
html.xpath('//span[@class="short"]/text()')

## 短评内容的分页抓取

抓取前十页的短评内容

In [ ]:
max_page = 10

In [ ]:
contents=[]
for page in range(1,max_page+1):
    url = f'https://movie.douban.com/subject/26654184/comments?start={(page-1)*20}&limit=20&status=P&sort=new_score'
    res=req.get(url,headers=headers)
    html=etree.HTML(res.text)
    content=html.xpath('//span[@class="short"]/text()')
    print(f'{page}/{max_page}-------{len(content)}')
    contents.extend(content)

In [ ]:
for i in contents:
    print(i+'\n')

## 抓取豆瓣影评页面中的其他内容

- 用户昵称
- 用户头像原始图片
- 用户评分登记
- 用户评论时间

### 用户昵称

In [ ]:
name=html.xpath('//span[@class="comment-info"]/a/text()')
name

### 用户头像原始图片

In [ ]:
img_url = html.xpath('//div[@class="avatar"]/a/img/@src')
img_url

### 用户评分登记

#### 按照元素出现的顺序筛选

In [ ]:
grade=html.xpath('//span[@class="comment-info"]/span[2]/@class')
grade

#### 使用正则表达式筛选

In [ ]:
pattern='<span class="(allstar\d0.*? rating)" title='
re.findall(pattern,res.text)

#### 使用Xpath表达式进行筛选

In [ ]:
#contains：对某个标签的属性值进行筛选，筛选出包含某个字符串的标签
grade=html.xpath('//span[contains(@class,"allstar")]/@class')
grade

### 用户评论时间

In [ ]:
time=html.xpath('//span[@class="comment-time "]/@title')
time

## 数据的存储

In [ ]:
len(content),len(name),len(img_url),len(grade),len(time)

In [ ]:
import pandas as pd

In [ ]:
#定义数据框
dataframe=pd.DataFrame([name,time,grade,content,img_url],
                       index=['用户昵称','评论时间','评论星级','评论内容','用户头像地址']).T
dataframe

In [ ]:
#存储为Excel文件
dataframe.to_excel('douban.xlsx')

## 修改分页抓取，获取所有字段并保存

In [ ]:
max_page=10
contents=[]
names=[]
imgs_url=[]
grades=[]
times=[]
for page in range(1,max_page+1):
    url = f'https://movie.douban.com/subject/26654184/comments?start={(page-1)*20}&limit=20&status=P&sort=new_score'
    res=req.get(url,headers=headers)
    html=etree.HTML(res.text)
    
    
    
    content=html.xpath('//span[@class="short"]/text()')
    name=html.xpath('//span[@class="comment-info"]/a/text()')
    img_url = html.xpath('//div[@class="avatar"]/a/img/@src')
    grade=html.xpath('//span[@class="comment-info"]/span[2]/@class')
    time=html.xpath('//span[@class="comment-info"]/span[3]/@title')
    contents.extend(content)
    names.extend(name)
    imgs_url.extend(img_url)
    grades.extend(grade)
    times.extend(time)

    
    print(f'{page}/{max_page}-------{len(content),len(name),len(img_url),len(grade),len(time)}')


In [ ]:
dataframe=pd.DataFrame([names,times,grades,contents,imgs_url],
                       index=['用户昵称','评论时间','评论星级','评论内容','用户头像地址']).T
dataframe

In [ ]:
dataframe.to_excel('douban.xlsx')

## 模拟登陆状态

In [ ]:
max_page=20
contents=[]
names=[]
imgs_url=[]
grades=[]
times=[]
for page in range(1,max_page+1):
    url = f'https://movie.douban.com/subject/26654184/comments?start={(page-1)*20}&limit=20&status=P&sort=new_score'
    res=req.get(url,headers=headers)
    html=etree.HTML(res.text)
    
    
    
    content=html.xpath('//span[@class="short"]/text()')
    name=html.xpath('//span[@class="comment-info"]/a/text()')
    img_url = html.xpath('//div[@class="avatar"]/a/img/@src')
    grade=html.xpath('//span[@class="comment-info"]/span[2]/@class')
    time=html.xpath('//span[@class="comment-info"]/span[3]/@title')
    contents.extend(content)
    names.extend(name)
    imgs_url.extend(img_url)
    grades.extend(grade)
    times.extend(time)

    
    print(f'{page}/{max_page}-------{len(content),len(name),len(img_url),len(grade),len(time)}')


发现在前11页访问时没有问题，但是12页开始已经是访问不到数据，打开第12页的网页地址发现，需要登陆才能查看

In [ ]:
headers = {
    'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:106.0) Gecko/20100101 Firefox/106.0',
    'Cookie':__import__('os').environ.get('SCRAPER_COOKIE', '')
    }

In [ ]:
res=req.get(url,headers=headers)

In [ ]:
res.status_code

In [ ]:
max_page=20
contents=[]
names=[]
imgs_url=[]
grades=[]
times=[]
for page in range(1,max_page+1):
    url = f'https://movie.douban.com/subject/26654184/comments?start={(page-1)*20}&limit=20&status=P&sort=new_score'
    res=req.get(url,headers=headers)
    html=etree.HTML(res.text)
    
    
    
    content=html.xpath('//span[@class="short"]/text()')
    name=html.xpath('//span[@class="comment-info"]/a/text()')
    img_url = html.xpath('//div[@class="avatar"]/a/img/@src')
    grade=html.xpath('//span[@class="comment-info"]/span[2]/@class')
    time=html.xpath('//span[@class="comment-time "]/@title')
    contents.extend(content)
    names.extend(name)
    imgs_url.extend(img_url)
    grades.extend(grade)
    times.extend(time)

    
    print(f'{page}/{max_page}-------{len(content),len(name),len(img_url),len(grade),len(time)}')


In [ ]:
dataframe=pd.DataFrame([names,times,grades,contents,imgs_url],
                       index=['用户昵称','评论时间','评论星级','评论内容','用户头像地址']).T
dataframe

In [ ]:
dataframe.to_excel('douban.xlsx')